In [1]:
import numpy as np
from tensorflow.keras.layers import Dense, Conv1D, MaxPooling1D, Flatten, Add, Input, Activation
from tensorflow.keras.models import Sequential, Model
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import scipy.io
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, f1_score

# load data from .mat file
filename = 'data_PY3.mat'
mat = scipy.io.loadmat(filename)
train_data = mat['X_n']
train_labels = mat['Y_n']

ListX = train_data
ListY = train_labels

# set random seed for reproducibility
np.random.seed(42)



In [ ]:
pip install matplotlib

C:\Users\PEZ37\AppData\Local\Temp\ipykernel_20156\2828990782.py:57: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasClassifier(build_fn=create_model, epochs=50, batch_size=32, verbose=0)


Fitting 6 folds for each of 1 candidates, totalling 6 fits


ValueError: 
All the 6 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\PEZ37\Anaconda3\envs\CNN\lib\site-packages\sklearn\model_selection\_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\PEZ37\Anaconda3\envs\CNN\lib\site-packages\keras\wrappers\scikit_learn.py", line 248, in fit
    return super().fit(x, y, **kwargs)
  File "C:\Users\PEZ37\Anaconda3\envs\CNN\lib\site-packages\keras\wrappers\scikit_learn.py", line 175, in fit
    history = self.model.fit(x, y, **fit_args)
  File "C:\Users\PEZ37\Anaconda3\envs\CNN\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
    raise e.with_traceback(filtered_tb) from 

In [2]:
num_classes = 6
# model 0: one-layer CNN
model0 = Sequential()
model0.add(Conv1D(filters=32, kernel_size=2, activation='relu', input_shape=(46, 1200)))
model0.add(MaxPooling1D(pool_size=2))
model0.add(Flatten())
model0.add(Dense(64, activation='relu'))
model0.add(Dense(num_classes, activation='softmax'))
model0.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# model 1: two-layer CNN
model1 = Sequential()
model1.add(Conv1D(filters=16, kernel_size=2, activation='relu', input_shape=(46, 1200)))
model1.add(MaxPooling1D(pool_size=2))
model1.add(Conv1D(filters=32, kernel_size=2, activation='relu'))
model1.add(MaxPooling1D(pool_size=2))
model1.add(Flatten())
model1.add(Dense(64, activation='relu'))
model1.add(Dense(num_classes, activation='softmax'))
model1.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# model 2: three-layer CNN
model2 = Sequential()
model2.add(Conv1D(filters=16, kernel_size=2, activation='relu', input_shape=(46, 1200)))
model2.add(MaxPooling1D(pool_size=2))
model2.add(Conv1D(filters=32, kernel_size=2, activation='relu'))
model2.add(MaxPooling1D(pool_size=2))
model2.add(Conv1D(filters=64, kernel_size=2, activation='relu'))
model2.add(MaxPooling1D(pool_size=2))
model2.add(Flatten())
model2.add(Dense(6, activation='relu'))
model2.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# model 3: residual CNN
def residual_module(layer_in, n_filters):
    merge_input = layer_in
    if layer_in.shape[-1] != n_filters:
        merge_input = Conv1D(n_filters, kernel_size=1, activation='relu', padding='same')(layer_in)
    conv1 = Conv1D(n_filters, kernel_size=2, activation='relu', padding='same')(layer_in)
    conv2 = Conv1D(n_filters, kernel_size=2, activation='linear', padding='same')(conv1)
    layer_out = Add()([conv2, merge_input])
    layer_out = Activation('relu')(layer_out)
    return layer_out

In [3]:
inputs = Input(shape=(46, 1200))
r1 = residual_module(inputs, 32)
r2 = residual_module(r1, 64)
r3 = residual_module(r2, 128)
pool = MaxPooling1D(pool_size=2)(r3)
flat = Flatten()(pool)
hidden1 = Dense(64, activation='relu')(flat)
output = Dense(num_classes, activation='softmax')(hidden1)
model3 = Model(inputs=inputs, outputs=output)
model3.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# train and evaluate the models
models = [model0, model1, model2, model3]
model_names = ['One-Layer CNN', 'Two-Layer CNN', 'Three-Layer CNN', 'Residual CNN']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# subset the data to the selected sensors
sensor_nums = [1, 2, 4, 11, 46]

models = []
layer_nums = [1, 2]
kernel_sizes = [10]
activations = ['relu', 'sigmoid', 'tanh']

sensors = 1
posi = range(0,46,sensors)
length = len(posi)
Listx = np.zeros((150,46, 1200))
Listx[:, posi,:] = ListX[:,posi,:]
Listy = ListY
X = np.array(Listx)
y = np.array(Listy)

# fit the model and store results
def add_noise(data, snr_db):
    signal_power = np.mean(np.square(data))
    noise_power = signal_power / (5 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), data.shape)
    noisy_data = data + noise
    return noisy_data

snr_db = 100 # signal-to-noise ratio in dB
noisy_train_data = np.zeros(X.shape)
for i in range(X.shape[0]):
    noisy_train_data[i] = add_noise(X[i], snr_db)

# loop through hyperparameters and train models
accuracies = []
precisions = []
recalls = []
f1_scores = []
confusion_matrices = []
num_repeats = 200
for layer in layer_nums:
    for kernel_size in kernel_sizes:
        for activation in activations:
            print(f'Layers: {layer}, Kernel size: {kernel_size}, Activation: {activation}')      
            for i in range(num_repeats):
                X_train, X_test, y_train, y_test = train_test_split(noisy_train_data, y, test_size=0.4, random_state=43)
                y_train = to_categorical(y_train)
                y_test = to_categorical(y_test)
                # create model
                model = Sequential()
                for i in range(layer):
                    model.add(Conv1D(filters=32, kernel_size=kernel_size, activation=activation, input_shape=(46, 1200)))
                    model.add(MaxPooling1D(pool_size=2))
                model.add(Flatten())
                model.add(Dense(64, activation=activation))
                model.add(Dense(6, activation='softmax'))
                model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

                # train model and print results
                history = model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.16)
                test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

                y_pred = model.predict(X_test)
                y_pred = np.argmax(y_pred, axis=1)
                y_test = np.argmax(y_test, axis=1)

                precision = precision_score(y_test, y_pred, average='weighted')
                recall = recall_score(y_test, y_pred, average='weighted')
                f1 = f1_score(y_test, y_pred, average='weighted')
                conf_matrix = confusion_matrix(y_test, y_pred)

                accuracies.append(test_acc)
                precisions.append(precision)
                recalls.append(recall)
                f1_scores.append(f1)
                confusion_matrices.append(conf_matrix)
                

Layers: 1, Kernel size: 10, Activation: relu
2/2 [==============================] - 0s 5ms/step


In [ ]:
# Create a function to plot box plots for a given metric
def plot_boxplot(metric_data, metric_name):
    num_models = 6
    data_per_model = len(metric_data) // num_models
    model_data = [metric_data[i * data_per_model:(i + 1) * data_per_model] for i in range(num_models)]

    fig, ax = plt.subplots()
    positions = [i for i in range(1, num_models + 1)]
    boxprops = dict(edgecolor='black')
    ax.boxplot(model_data, positions=positions, boxprops=boxprops, widths=1, patch_artist=True, showfliers=False)
    for box in ax.artists:
        box.set_facecolor(box_color)
    ax.set_xlabel("CNN Model", fontsize=15, fontname='Calibri')
    ax.set_ylabel(metric_name, fontsize=15, fontname='Calibri')
    plt.show()
    fig.savefig(f'myplot_{metric_name}.png', dpi=300)

# Plot box plots for accuracy, precision, recall, and F1-score

plot_boxplot(accuracies, "Accuracy")
plot_boxplot(precisions, "Precision")
plot_boxplot(recalls, "Recall")
plot_boxplot(f1_scores, "F1-score")

# Get the index of the best model based on the highest F1-score
best_model_index = np.argmax(f1_scores)

# Plot the confusion matrix for the best model
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrices[best_model_index], annot=True, cmap='viridis', fmt='d', cbar=False)
plt.xlabel("Predicted Labels", fontsize=15, fontname='Calibri')
plt.ylabel("True Labels", fontsize=15, fontname='Calibri')
plt.title("Confusion Matrix for the Best Model", fontsize=18, fontname='Calibri')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# subset the data to the selected sensors
models = []
layer_nums = [1]
kernel_sizes = [10]
activations = ['tanh']

sensors = 4
posi = range(0,46,sensors)

length = len(posi)
print(length)
Listx = np.zeros((150,46, 1200))
Listx[:, posi,:] = ListX[:,posi,:]
Listy = ListY
X = np.array(Listx)
y = np.array(Listy)

# fit the model and store results
def add_noise(data, snr_db):
    signal_power = np.mean(np.square(data))
    noise_power = signal_power / (5 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), data.shape)
    noisy_data = data + noise
    return noisy_data

snr_db = 5 # signal-to-noise ratio in dB
noisy_train_data = np.zeros(X.shape)
for i in range(X.shape[0]):
    noisy_train_data[i] = add_noise(X[i], snr_db)

# loop through hyperparameters and train models
accuracies = []
precisions = []
recalls = []
f1_scores = []
confusion_matrices = []
num_repeats = 1
for layer in layer_nums:
    for kernel_size in kernel_sizes:
        for activation in activations:
            print(f'Layers: {layer}, Kernel size: {kernel_size}, Activation: {activation}')      
            for i in range(num_repeats):
                X_train, X_test, y_train, y_test = train_test_split(noisy_train_data, y, test_size=0.4, random_state=43)
                y_train = to_categorical(y_train)
                y_test = to_categorical(y_test)
                # create model
                model = Sequential()
                for i in range(layer):
                    model.add(Conv1D(filters=32, kernel_size=kernel_size, activation=activation, input_shape=(46, 1200)))
                    model.add(MaxPooling1D(pool_size=2))
                model.add(Flatten())
                model.add(Dense(64, activation=activation))
                model.add(Dense(6, activation='softmax'))
                model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

                # train model and print results
                history = model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0, validation_split=0.16)
                test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

                x_test = noisy_train_data
                y_pred = model.predict(x_test)
                y_pred = np.argmax(y_pred, axis=1)
                y_test = y
                print(y_pred)
                print(y_test)

                precision = precision_score(y_test, y_pred, average='weighted')
                recall = recall_score(y_test, y_pred, average='weighted')
                f1 = f1_score(y_test, y_pred, average='weighted')
                conf_matrix = confusion_matrix(y_test, y_pred)
               
                accuracies.append(test_acc)
                precisions.append(precision)
                recalls.append(recall)
                f1_scores.append(f1)
                confusion_matrices.append(conf_matrix)
                

In [ ]:
# Plot box plots for accuracy, precision, recall, and F1-score

print(accuracies, "Accuracy")
print(precisions, "Precision")
print(recalls, "Recall")
print(f1_scores, "F1-score")

# Get the index of the best model based on the highest F1-score
best_model_index = np.argmax(f1_scores)

# Plot the confusion matrix for the best model
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrices[best_model_index], annot=True, cmap='viridis', fmt='d', cbar=False)
plt.xlabel("Predicted Labels", fontsize=15, fontname='Calibri')
plt.ylabel("True Labels", fontsize=15, fontname='Calibri')
plt.title("Confusion Matrix for the Best Model", fontsize=18, fontname='Calibri')
plt.show()